# Enveda CASMI 2026 — inference-only notebook (frozen deep MLP)
Loads the artifacts frozen by the training notebook (`frozen/`: trained weights, training-column selection, feature settings, calibration offsets and the
training part of the candidate library), adds the public COCONUT cache and the PubChem popularity prior, and writes `submission.csv`.
It does no training and no canonicalisation, so it runs in minutes, and it is the notebook to submit for experiments on the **pool and ranking**
(prior weight, extra pools) and for the hidden-test rerun.

**Inputs to attach:** the competition data; the dataset made from the training notebook's `frozen/` folder; the COCONUT cache dataset; the
popularity dataset (`dmitriigluzdov/casmi26-pubchem-popularity-prior`, only needed when `POP_WEIGHT > 0`). A GPU is not needed.

In [ ]:
# ======================================== 1. CONFIG (inference only) ========================================
import os, sys, glob, time, json, gc, queue, threading, warnings, re
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
T0 = time.time()
def rss_gb():
    try:
        import psutil; return psutil.Process().memory_info().rss / 1e9
    except Exception: return float("nan")
def tick(msg): print(f"[{(time.time() - T0) / 60:6.1f} min | {rss_gb():4.1f} GB] {msg}", flush=True)

DEBUG = bool(int(os.environ.get("CASMI_DEBUG", "0")))   # only for local testing against a frozen DEBUG run

# ############################################ SETTINGS: THE ONLY LINES TO EDIT ############################################
# Values to use come from docs/EXPERIMENT_LOG.md (entry "P7d"). Leaderboard history: train-only 0.076 | + COCONUT 0.128 | + prior w=0.15 0.146 | w=0.30 0.140.
POP_WEIGHT = 0.15            # popularity prior weight. score = z(cosine within the mass window) + POP_WEIGHT * pop.  0.0 = off (pure cosine).
USE_TIER = False             # True = add the PubChem mass-window channel (P7d). Needs POP_WEIGHT > 0 and the popularity dataset (it holds the tier arrays).
TIER_LIB_BONUS = 0.0         # only with USE_TIER: source prior added to every library (train + COCONUT) candidate, in z-score units. Higher = trust the library more.
TIER_WORKERS = 4             # processes for fingerprinting / tautomer keys of the tier candidates (Kaggle gives 4 cores)
WINDOW_SCALE = 1.0           # multiplies every per-adduct mass-window half-width (frozen: max(3 ppm, 3.5 sigma) fitted on enveda-180). 1.0 = as trained. >1 probes whether the hidden test needs wider windows.
CENTRE_ON_OFFSET = True      # True: windows centred on the per-adduct instrument offset fitted on enveda-180 (+2.2 ppm for [M+H]+). False: centred on 0 (v0.1 style).
EXTRA_POOL_DIR = "auto"      # COCONUT cache (coconut_v1.parquet + coconut_fp_v1.npy): "auto" = search /kaggle/input/** then ./artifacts. None = train-only library.
# ##########################################################################################################################
POP_PATH = "auto"            # pool_popularity.csv ("auto": search /kaggle/input/** then ./data/external/pop_prior)
FROZEN_DIR = "auto"          # folder with frozen_meta.json ("auto": search /kaggle/input/**, then $CASMI_FROZEN_DIR, then ./kaggle_work/frozen)
if os.environ.get("CASMI_POP_WEIGHT"): POP_WEIGHT = float(os.environ["CASMI_POP_WEIGHT"])      # (override used for local testing)
if os.environ.get("CASMI_USE_TIER"): USE_TIER = bool(int(os.environ["CASMI_USE_TIER"]))
if os.environ.get("CASMI_WINDOW_SCALE"): WINDOW_SCALE = float(os.environ["CASMI_WINDOW_SCALE"])
if os.environ.get("CASMI_TIER_LIB_BONUS"): TIER_LIB_BONUS = float(os.environ["CASMI_TIER_LIB_BONUS"])
SEED = 42

def find_comp_dir():
    if os.environ.get("CASMI_COMP_DIR"): return os.environ["CASMI_COMP_DIR"]
    hits = glob.glob("/kaggle/input/**/test.parquet", recursive=True)
    return os.path.dirname(hits[0]) if hits else "./data"
COMP_DIR = find_comp_dir()
TEST_PATH, SAMPLE_PATH = (os.path.join(COMP_DIR, f) for f in ("test.parquet", "sample_submission.csv"))
WORK = os.environ.get("CASMI_WORK") or ("/kaggle/working" if os.path.isdir("/kaggle/working") else "./kaggle_work")
os.makedirs(WORK, exist_ok=True)
if FROZEN_DIR == "auto":
    hits = glob.glob("/kaggle/input/**/frozen_meta.json", recursive=True)
    FROZEN_DIR = os.path.dirname(hits[0]) if hits else (os.environ.get("CASMI_FROZEN_DIR") or "./kaggle_work/frozen")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, "| DEBUG =", DEBUG, "| data:", COMP_DIR, "| frozen:", FROZEN_DIR, "| work:", WORK)
print(f"SETTINGS: POP_WEIGHT={POP_WEIGHT} | USE_TIER={USE_TIER} | TIER_LIB_BONUS={TIER_LIB_BONUS if USE_TIER else 'n/a'} | WINDOW_SCALE={WINDOW_SCALE} | CENTRE_ON_OFFSET={CENTRE_ON_OFFSET} | EXTRA_POOL_DIR={EXTRA_POOL_DIR!r}")
if USE_TIER:
    assert POP_WEIGHT > 0, "USE_TIER needs POP_WEIGHT > 0: without the prior, thousands of same-mass PubChem isomers swamp the library"
    _need = ["pc_mass", "pc_off", "pc_smiles", "pc_ik14", "pc_lsid", "pc_lpmid"]
    _root = os.environ.get("CASMI_TIER_DIR") or next((os.path.dirname(h) for h in glob.glob("/kaggle/input/**/pc_lsid.npy", recursive=True) + glob.glob("data/external/pubchem_tier/pc_lsid.npy")), None)
    _missing = [n for n in _need if _root is None or not glob.glob(f"{_root}/**/{n}.npy", recursive=True)]
    assert not _missing, f"PubChem tier arrays missing {_missing}: attach dmitriigluzdov/casmi26-pubchem-popularity-prior (searched {_root})"
    print("PubChem tier found under", _root)
assert os.path.exists(TEST_PATH), f"test.parquet not found under {COMP_DIR}"
assert os.path.exists(os.path.join(FROZEN_DIR, "frozen_meta.json")), f"no frozen_meta.json in {FROZEN_DIR}: attach the dataset made from the training notebook's frozen/ folder"


In [ ]:
# ======================================== 2. MASS HELPERS ========================================
# ion m/z = neutral mass + shift. [M+K]+ and [M+Cl]- appear in the test file (4 spectra) but not in the v0.1 cleaning table.
SHIFT = {"[M+H]+": 1.007276, "[M-H]-": -1.007276, "[M+Na]+": 22.989218, "[M+NH4]+": 18.033823,
         "[M+CH2O2-H]-": 44.997655, "[M+K]+": 38.963158, "[M+Cl]-": 34.969402}
V01_ADDUCTS = ["[M+H]+", "[M-H]-", "[M+Na]+", "[M+NH4]+", "[M+CH2O2-H]-"]     # cleaning table (dimers excluded)

import re
_FORMULA, _fm_cache = re.compile(r"([A-Z][a-z]?)(\d*)"), {}
def formula_mass(formula):
    """Monoisotopic mass of a neutral formula string such as 'C20H28N2O3' (NaN if unparsable)."""
    if formula in _fm_cache: return _fm_cache[formula]
    pt = Chem.GetPeriodicTable()
    try:
        if not re.fullmatch(r"([A-Z][a-z]?\d*)+", formula): raise ValueError
        m = sum(pt.GetMostCommonIsotopeMass(el) * (int(n) if n else 1) for el, n in _FORMULA.findall(formula))
    except Exception:
        m = np.nan
    _fm_cache[formula] = m
    return m

def fit_bias(err_ppm, adduct):
    """Per-adduct calibration offset (median signed error, ppm) and robust spread (1.4826 * MAD)."""
    g = pd.DataFrame({"err": err_ppm, "adduct": adduct}).groupby("adduct").err
    return pd.DataFrame({"n": g.size(), "bias_ppm": g.median(), "sd_ppm": g.apply(lambda s: 1.4826 * np.median(np.abs(s - s.median())))})

def halfwidth_by_adduct(tab, k=3.5, floor=3.0):
    """Per-adduct half-width in ppm: k robust sigmas, never below `floor`."""
    return (k * tab.sd_ppm).clip(lower=floor).to_dict()

def mass_window(nm, adduct, tol_ppm, bias):
    """Candidate-mass interval [lo, hi] per query: window centred on the offset-corrected neutral mass nm / (1 + offset)."""
    nm = np.asarray(nm, float)
    nm = nm / (1 + pd.Series(adduct).map(bias).fillna(0.0).values * 1e-6)
    tol = pd.Series(adduct).map(tol_ppm).fillna(max(tol_ppm.values())).values
    return nm * (1 - tol * 1e-6), nm * (1 + tol * 1e-6)

class Library:
    """Candidate pool: unit-length fingerprints for cosine ranking and a mass-sorted index for the window search."""
    def __init__(self, smiles, keys, mass, fp_unit):
        self.smiles, self.keys, self.mass, self.fp_unit = smiles, np.asarray(keys), np.asarray(mass), fp_unit
        self.order = np.argsort(self.mass); self.mass_sorted = self.mass[self.order]
        self.pop, self.pop_w = None, 0.0        # optional popularity prior (one value per library row) and its weight; None = pure cosine

def unit(A): return A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)


In [ ]:
# ======================================== 6. SPECTRUM FEATURES ========================================
# WHY (project findings P4a-c): features = fragment peaks + NEUTRAL LOSSES (precursor m/z - fragment m/z), each binned as
# nominal mass x 0.02 Da MASS-DEFECT sub-bins (the digits after the decimal point carry elemental composition), sqrt intensity,
# unit length per block, plus acquisition metadata. Sparse matrices (about 130 non-zeros per spectrum).
import scipy.sparse as sp
import pyarrow.parquet as pq
MAX_MZ, DEFECT_LO, DEFECT_HI, W = 1000, -0.05, 0.25, 0.02
K_SUB = int(round((DEFECT_HI - DEFECT_LO) / W)); PER = K_SUB + 1          # 15 sub-bins + 1 overflow column per nominal mass

def defect_block(axes, its, nb_nominal=MAX_MZ, eps=1e-6):
    """Nominal n = floor(x + 0.05); defect d = x - n; d in [-0.05, 0.25) -> one of 15 sub-bins, else an overflow column."""
    r_, c_, v_ = [], [], []
    for r, (a, i) in enumerate(zip(axes, its)):
        a = np.asarray(a, float); n = np.floor(a + 0.05 + eps).astype(int); d = a - n
        ok = (a >= 0) & (n < nb_nominal)
        if not ok.any(): continue
        sub = np.where((d >= DEFECT_LO) & (d < DEFECT_HI), np.floor((d - DEFECT_LO) / W + eps).astype(int).clip(0, K_SUB - 1), K_SUB)
        u, inv = np.unique((n * PER + sub)[ok], return_inverse=True)
        v = np.bincount(inv, weights=np.sqrt(np.asarray(i, float)[ok])); v /= np.linalg.norm(v) + 1e-9
        r_.append(np.full(len(u), r)); c_.append(u); v_.append(v)
    if not r_: return sp.csr_matrix((len(axes), nb_nominal * PER), dtype=np.float32)
    return sp.csr_matrix((np.concatenate(v_).astype(np.float32), (np.concatenate(r_), np.concatenate(c_))), shape=(len(axes), nb_nominal * PER))

def ce_scalar(x):
    """Collision energy: list -> mean and flag 1 (every row stores a list; the flag therefore means 'has a CE value')."""
    if x is None: return np.nan, 0.0
    a = np.asarray(x, float); return (np.nan if a.size == 0 else float(np.nanmean(a))), 1.0

class Features:
    def __init__(self, fit_df):
        self.top_inst = fit_df.instrument_type.value_counts().head(8).index
        self.cols = self._meta(fit_df).columns; self.ce_med = fit_df.collision_energy_ev.median()
    def _meta(self, d):
        m = d[["ionization_mode", "instrument_type", "adduct"]].copy()
        m["instrument_type"] = m.instrument_type.where(m.instrument_type.isin(self.top_inst), "other")
        return pd.get_dummies(m)
    def build(self, d, mzs, its):
        frag = defect_block(mzs, its)
        loss = defect_block([np.where(p - np.asarray(m) >= 1.0, p - np.asarray(m), -1.0) for m, p in zip(mzs, d.precursor_mz.values)], its)
        M = self._meta(d).reindex(columns=self.cols, fill_value=0).values.astype(np.float32)
        num = np.c_[d.neutral_mass / MAX_MZ, d.collision_energy_ev.fillna(self.ce_med) / 100, d.ce_multi].astype(np.float32)
        return sp.hstack([frag, loss, sp.csr_matrix(np.hstack([num, M]))], format="csr")

def with_meta(frame, t):
    """Attach the per-spectrum acquisition columns read from the parquet (t = pandas frame with the same length/order)."""
    d = frame[["adduct", "precursor_mz"]].reset_index(drop=True).copy()
    d["instrument_type"], d["ionization_mode"] = t.instrument_type.values, t.ionization_mode.values
    ce = [ce_scalar(x) for x in t.collision_energy_ev]
    d["collision_energy_ev"], d["ce_multi"] = [c[0] for c in ce], np.float32([c[1] for c in ce])
    d["neutral_mass"] = d.precursor_mz - d.adduct.map(SHIFT)               # n_molecules = 1 for every monomer adduct
    return d

def read_rows_meta(row_ids):
    """instrument_type / ionization_mode / collision energy for selected rows (small columns only)."""
    pf = pq.ParquetFile(TRAIN_PATH); starts = np.cumsum([0] + [pf.metadata.row_group(i).num_rows for i in range(pf.num_row_groups)])
    row_ids = np.asarray(row_ids); parts = []
    for g in range(pf.num_row_groups):
        m = np.flatnonzero((row_ids >= starts[g]) & (row_ids < starts[g + 1]))
        if len(m):
            t = pf.read_row_group(g, columns=["instrument_type", "ionization_mode", "collision_energy_ev"]).take(row_ids[m] - starts[g]).to_pandas()
            t.index = m; parts.append(t)
    return pd.concat(parts).sort_index().reset_index(drop=True)


In [ ]:
# ======================================== 10. DEEP MODEL: RESIDUAL MLP ========================================
# sparse spectrum features -> Linear(1024) -> 4 pre-norm residual blocks (LayerNorm) -> 1,024 sigmoid outputs, trained with BCE on the
# Morgan bits. Early stopping on a scaffold-grouped 5% holdout INSIDE the training rows (the sanity holdout is never touched).
# Project finding: with ~10x more data than the first experiments this model overtakes Ridge by a wide margin.
import torch.nn as nn
class ResBlock(nn.Module):
    def __init__(self, d, dropout):
        super().__init__(); self.norm = nn.LayerNorm(d)
        self.f = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d, d))
    def forward(self, x): return x + self.f(self.norm(x))

def make_net(n_in, d=1024, blocks=4, dropout=0.3, n_out=1024):
    return nn.Sequential(nn.Linear(n_in, d), nn.ReLU(), nn.Dropout(dropout), *[ResBlock(d, dropout) for _ in range(blocks)], nn.LayerNorm(d), nn.Linear(d, n_out))

def batches(X, Y, index_batches, device):
    """Background thread densifies the sparse rows while the accelerator works on the previous batch."""
    q = queue.Queue(maxsize=6)
    def work():
        for b in index_batches:
            xb = torch.from_numpy(X[b].toarray()); yb = torch.from_numpy(Y[b]) if Y is not None else None
            if device == "cuda": xb = xb.pin_memory()
            q.put((xb, yb))
        q.put(None)
    threading.Thread(target=work, daemon=True).start()
    while True:
        item = q.get()
        if item is None: return
        yield item[0].to(device), (item[1].to(device) if item[1] is not None else None)

def fit_mlp(X, Y, scaffolds, max_epochs, patience, lr=1e-3, weight_decay=1e-2, batch_size=256, seed=SEED, ckpt=None):
    torch.manual_seed(seed); np.random.seed(seed)
    fit_i, es_i = next(GroupShuffleSplit(n_splits=1, test_size=0.05, random_state=seed).split(X, groups=scaffolds))
    Xf, Xe, Yf, Ye = X[fit_i], X[es_i], Y[fit_i], Y[es_i]
    net = make_net(X.shape[1]).to(DEVICE); opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=weight_decay)
    lossf = nn.BCEWithLogitsLoss(); rng = np.random.default_rng(seed); best, best_state, bad = 1e9, None, 0
    for ep in range(max_epochs):
        t0 = time.time(); net.train(); perm = rng.permutation(len(fit_i)); tr_loss = 0.0
        for xb, yb in batches(Xf, Yf, [perm[i:i + batch_size] for i in range(0, len(perm), batch_size)], DEVICE):
            opt.zero_grad(); loss = lossf(net(xb), yb.float()); loss.backward(); opt.step(); tr_loss += loss.item() * len(xb)
        net.eval(); es = 0.0
        with torch.no_grad():
            for xb, yb in batches(Xe, Ye, [np.arange(i, min(i + 1024, Xe.shape[0])) for i in range(0, Xe.shape[0], 1024)], DEVICE):
                es += lossf(net(xb), yb.float()).item() * len(xb)
        es /= Xe.shape[0]; tr_loss /= len(fit_i)
        tick(f"  epoch {ep:2d} | train BCE {tr_loss:.5f} | inner-holdout BCE {es:.5f} | {time.time() - t0:.0f}s")
        if es < best - 1e-5:
            best, bad, best_state = es, 0, {k: v.detach().clone() for k, v in net.state_dict().items()}
            if ckpt: torch.save(best_state, ckpt)
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(best_state); return net

def predict_mlp(net, X, bs=1024):
    net.eval(); out = []
    with torch.no_grad():
        for xb, _ in batches(X, None, [np.arange(i, min(i + bs, X.shape[0])) for i in range(0, X.shape[0], bs)], DEVICE):
            out.append(torch.sigmoid(net(xb)).cpu().numpy())
    return np.vstack(out).astype(np.float32)


In [ ]:
# ======================================== 4b. EXTERNAL POOL + POPULARITY PRIOR (shared by the training and the inference notebook) ========================================
def merge_coconut(lib_smiles, lib_keys, fp_u8, mass):
    """Append the pre-computed COCONUT cache (EXTRA_POOL_DIR) to the library arrays. Structures already in the library win; one row per canonical key."""
    d_ = EXTRA_POOL_DIR
    if not d_: return lib_smiles, lib_keys, fp_u8, mass
    if d_ == "auto":
        hits = glob.glob("/kaggle/input/**/coconut_v1.parquet", recursive=True) + glob.glob("artifacts/coconut_v1.parquet")
        d_ = os.path.dirname(hits[0]) if hits else None
    if d_ is None:
        print("WARNING: no COCONUT cache found -> library = training molecules only (expect a score near 0 on novel molecules)")
        return lib_smiles, lib_keys, fp_u8, mass
    C_ = pd.read_parquet(f"{d_}/coconut_v1.parquet"); cfp = np.load(f"{d_}/coconut_fp_v1.npy", mmap_mode="r")
    assert len(C_) == len(cfp) and cfp.shape[1] == 1024, "COCONUT cache rows do not match its fingerprints"
    keep = ~C_.key.isin(set(lib_keys)).values & ~C_.key.duplicated().values        # training structures win; one row per canonical key
    idx_ = np.flatnonzero(keep); C_ = C_.iloc[idx_]
    out = (list(lib_smiles) + C_.smiles.tolist(), list(lib_keys) + C_.key.tolist(), np.vstack([fp_u8, np.asarray(cfp[idx_])]), np.concatenate([mass, C_.mass.values]))
    tick(f"+ {len(C_):,} COCONUT candidates from {d_}")
    return out

def attach_popularity(LIB, weight):
    """P7c: PubChem documentation prior, ln(1+substances)+ln(1+PubMed)+..., joined on the metric key; unmatched keys get the median (neutral)."""
    hits = glob.glob("/kaggle/input/**/pool_popularity.csv", recursive=True) + glob.glob("data/external/pop_prior/pool_popularity.csv")
    path = hits[0] if POP_PATH == "auto" and hits else (POP_PATH if POP_PATH != "auto" else None)
    if path is None: raise FileNotFoundError("POP_WEIGHT > 0 but pool_popularity.csv was not found: attach the dataset dmitriigluzdov/casmi26-pubchem-popularity-prior")
    pp = pd.read_csv(path, usecols=["inchikey14", "pop"]).drop_duplicates("inchikey14").set_index("inchikey14")["pop"]
    pop_ = pp.reindex(LIB.keys).values.astype(np.float32); miss_ = np.isnan(pop_)
    pop_[miss_] = np.nanmedian(pop_)
    LIB.pop, LIB.pop_w = pop_, float(weight)
    tick(f"popularity prior ON: weight {weight}, {1 - miss_.mean():.1%} of the library matched, pop median {np.median(pop_):.2f}")


In [ ]:
# ======================================== 2. LOAD THE FROZEN MODEL SETTINGS AND THE LIBRARY ========================================
fd = FROZEN_DIR; META = json.load(open(f"{fd}/frozen_meta.json"))
BIAS, HW = META["bias"], META["hw"]                                                        # per-adduct calibration offsets / half-widths (ppm)
HW = {k: v * WINDOW_SCALE for k, v in HW.items()}
if not CENTRE_ON_OFFSET: BIAS = {k: 0.0 for k in BIAS}
if WINDOW_SCALE != 1.0 or not CENTRE_ON_OFFSET: print(f"mass windows changed: half-widths x{WINDOW_SCALE} -> {({k: round(v, 1) for k, v in HW.items()})}, centred on offset: {CENTRE_ON_OFFSET}")
FEAT = Features.__new__(Features)                                                          # feature settings fitted on the training rows
FEAT.top_inst, FEAT.cols, FEAT.ce_med = pd.Index(META["top_inst"]), pd.Index(META["cols"]), META["ce_med"]
T_ = pd.read_parquet(f"{fd}/train_library.parquet"); fp_u8 = np.load(f"{fd}/train_library_fp.npy")
assert len(T_) == len(fp_u8) == META["n_library"], "frozen library rows do not match its fingerprints"
lib_smiles, lib_keys, mass = T_.smiles.tolist(), T_.key.tolist(), T_.mass.values
tick(f"frozen: {META['n_in']:,} input columns, {len(T_):,} training molecules, DEBUG-run = {META['debug']}")
lib_smiles, lib_keys, fp_u8, mass = merge_coconut(lib_smiles, lib_keys, fp_u8, mass)
LIB = Library(lib_smiles, lib_keys, mass, unit(fp_u8.astype(np.float32)))
if POP_WEIGHT > 0: attach_popularity(LIB, POP_WEIGHT)
tick(f"library ready: {len(LIB.keys):,} molecules, mass {LIB.mass_sorted[0]:.1f}-{LIB.mass_sorted[-1]:.1f}")


In [ ]:
# ======================================== 7. TEST SPECTRA ========================================
test = pd.read_parquet(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)
assert set(sample.molecule_id) == set(test.molecule_id), "sample_submission ids differ from the test file"
td = with_meta(test, test)                     # test has the same columns (adduct, precursor_mz, instrument_type, ionization_mode, collision_energy_ev)
X_TEST = FEAT.build(td, test.ms2_mzs.values, test.ms2_normalized_intensities.values).tocsr()
test_lo, test_hi = mass_window(td.neutral_mass.values, test.adduct.values, HW, BIAS)
tick(f"test: {len(test):,} spectra, {test.molecule_id.nunique():,} molecules; adducts {test.adduct.value_counts().to_dict()}")
print("spectra with an adduct that has no calibration offset (window centred on zero):", int((~test.adduct.isin(BIAS)).sum()))


In [ ]:
# ======================================== 8. RANKING + SUBMISSION WRITER ========================================
def rank_queries(Q, lo, hi, lib, k=25):
    """Q: unit-length predicted fingerprints (n x 1024); lo/hi: mass windows. Returns one array of library row indices per query,
    best first (<= k): library molecules whose exact mass lies in [lo, hi], ordered by cosine similarity to the query.
    Empty window -> widen around the window centre (10..250 ppm), finally fall back to the k nearest masses (a row must never be empty)."""
    out, ncand = [], np.zeros(len(Q), int)
    for i in range(len(Q)):
        a, b = np.searchsorted(lib.mass_sorted, [lo[i], hi[i]]); idx = lib.order[a:b]
        if len(idx) == 0:
            c = 0.5 * (lo[i] + hi[i])
            for ppm in (10, 25, 50, 100, 250):
                a, b = np.searchsorted(lib.mass_sorted, [c * (1 - ppm * 1e-6), c * (1 + ppm * 1e-6)])
                if b > a: idx = lib.order[a:b]; break
            if len(idx) == 0: idx = lib.order[np.argsort(np.abs(lib.mass_sorted - c))[:k]]
        ncand[i] = len(idx)
        s = lib.fp_unit[idx] @ Q[i]
        if lib.pop is not None and len(idx) > 1: s = (s - s.mean()) / (s.std() + 1e-6) + lib.pop_w * lib.pop[idx]      # P7c prior (off when lib.pop is None)
        if len(idx) > k:
            part = np.argpartition(-s, k)[:k]; order = part[np.argsort(-s[part])]
        else:
            order = np.argsort(-s)
        out.append(idx[order])
    return out, ncand

def pool_molecules(P, mol_ids, lo, hi):
    """One query per molecule_id: mean of the predicted fingerprints of its spectra (unit length) and the INTERSECTION of the spectra's
    mass windows (union if they do not overlap). Validated in the project: pooling beats scoring spectra one by one (+0.03 to +0.06 MRR)."""
    ids = pd.Series(mol_ids); groups = ids.groupby(ids).indices; keys = list(groups)
    Pm = unit(np.stack([P[groups[k]].mean(0) for k in keys]))
    L = np.array([lo[groups[k]].max() for k in keys]); H = np.array([hi[groups[k]].min() for k in keys])
    bad = L > H
    L[bad] = np.array([lo[groups[k]].min() for k in keys])[bad]; H[bad] = np.array([hi[groups[k]].max() for k in keys])[bad]
    return keys, Pm, L, H, int(bad.sum())

def validate_submission(sub, expected_ids):
    """The competition's rejection rules: columns present, non-empty, no nulls, no repeated molecule_id, <= 25 guesses per row."""
    assert list(sub.columns) == ["molecule_id", "smiles"], "need exactly the columns molecule_id,smiles"
    assert len(sub) > 0, "empty submission"
    assert sub.molecule_id.notna().all() and sub.smiles.notna().all(), "nulls present"
    assert (sub.smiles.astype(str).str.len() > 0).all(), "empty smiles field"
    assert sub.molecule_id.is_unique, "repeated molecule_id"
    assert set(sub.molecule_id) == set(expected_ids), "molecule_ids differ from the test set"
    n = sub.smiles.str.split(";").str.len()
    assert n.max() <= 25, f"more than 25 guesses in a row (max {n.max()})"
    return n

def publish(sub, label):
    """Write <WORK>/submission.csv: the file Kaggle picks up automatically when the notebook is submitted. Called as soon as the first model
    finishes (so a valid file always exists) and again when the final model is ready. Validated against the competition's rejection rules."""
    path = f"{WORK}/submission.csv"; sub.to_csv(path, index=False)
    validate_submission(pd.read_csv(path), sample.molecule_id)
    tick(f"submission.csv written from: {label}" + ("   (DEBUG run: tiny library, do NOT submit this one)" if DEBUG else ""))

def make_submission(P_test, name):
    """Predicted fingerprints per test SPECTRUM -> submission with one row per molecule_id."""
    keys, Pm, L, H, nbad = pool_molecules(P_test, test.molecule_id.values, test_lo, test_hi)
    ranked, ncand = rank_queries(Pm, L, H, LIB)
    sub = pd.DataFrame({"molecule_id": keys, "smiles": [";".join(LIB.smiles[j] for j in r) for r in ranked]})
    sub = sub.set_index("molecule_id").loc[sample.molecule_id].reset_index()            # same order as sample_submission
    n = validate_submission(sub, sample.molecule_id)
    path = f"{WORK}/submission_{name}{'_debug' if DEBUG else ''}.csv"; sub.to_csv(path, index=False)
    tick(f"{path}: {len(sub)} molecules | guesses per molecule: median {n.median():.0f}, min {n.min()} | candidates in window: median {np.median(ncand):.0f}, "
         f"empty windows {int((ncand == 0).sum())} | spectra windows that did not overlap: {nbad}")
    return sub, ranked, Pm

# holdout sanity check = the project's benchmark recipe: an ORACLE library (the true molecule is a candidate), per-spectrum queries,
# enveda-180 spectra only (the test instrument). Expect clearly non-zero scores; a score near 0 means a bug.
def holdout_sanity(P_hold, name, max_queries=3000):
    h = deep[deep.split == "holdout"].reset_index(drop=True)
    sel = np.flatnonzero((h.ingest_lib == "enveda-180").values)
    if len(sel) == 0: sel = np.arange(len(h))
    sel = sel[:max_queries]; hh = h.iloc[sel]
    nm = hh.precursor_mz.values - hh.adduct.map(SHIFT).values
    lo, hi = mass_window(nm, hh.adduct.values, HW, BIAS)
    ranked, ncand = rank_queries(unit(P_hold[sel]), lo, hi, LIB)
    rr = np.array([1 / (list(LIB.keys[r]).index(k) + 1) if k in set(LIB.keys[r]) else 0.0 for r, k in zip(ranked, hh.key.values)])
    tick(f"[{name}] holdout sanity (oracle library, {len(sel):,} enveda-180 spectra): MRR@25 = {rr.mean():.4f} | median candidates {np.median(ncand):.0f}")
    return rr.mean()


In [ ]:
# ======================================== P7d. PUBCHEM MASS-WINDOW CHANNEL ========================================
# WHY: ~70% of the hidden-test molecules are in neither the training set nor COCONUT (P7b/P7c leaderboard decomposition). PubChem (105.9M structures, sorted by
# neutral mass, with aligned documentation counts) is the pool that can contain them. We never fingerprint the whole tier: only the structures inside each
# query's mass window (a few hundred to a few thousand), on the fly, and rank them together with the library candidates by
#     z(cosine within the window) + w * pop,       pop = ln(1 + substance records) + ln(1 + PubMed articles)
# (the same prior as the library's pool_popularity.csv, so the two sources are comparable). Tier rows whose raw InChIKey14 is already a library key are
# skipped (the library copy wins). The final top list is deduplicated on the METRIC key (tautomer-canonical InChIKey14), computed for tier rows only
# for the best `topn` candidates (tautomer canonicalisation is ~20 ms per molecule).
import multiprocessing as mp
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")                                  # PubChem has structures RDKit cannot parse (e.g. odd valences): they are skipped, not worth a log line

class Tier:
    """Memory-mapped PubChem tier: sorted neutral masses, SMILES bytes + offsets, documentation counts, raw InChIKey14."""
    def __init__(self, root):
        def find(name):
            for pat in (f"{root}/{name}.npy", f"{root}/**/{name}.npy"):
                hits = glob.glob(pat, recursive=True)
                if hits: return hits[0]
            raise FileNotFoundError(f"{name}.npy not found under {root}")
        ld = lambda n: np.load(find(n), mmap_mode="r")
        self.mass, self.off, self.smi, self.lsid, self.lpmid, self.ik = (ld(n) for n in ("pc_mass", "pc_off", "pc_smiles", "pc_lsid", "pc_lpmid", "pc_ik14"))
        n = len(self.mass); assert len(self.off) == n + 1 and len(self.lsid) == len(self.lpmid) == len(self.ik) == n, "tier arrays are not row-aligned"
    def window(self, lo, hi): return tuple(int(x) for x in np.searchsorted(self.mass, [lo, hi]))
    def smiles(self, rows): return [bytes(self.smi[self.off[r]:self.off[r + 1]]).decode() for r in rows]
    def pop(self, rows): return self.lsid[rows].astype(np.float32) + self.lpmid[rows].astype(np.float32)
    def ik14(self, rows): return [x.decode() if isinstance(x, bytes) else str(x) for x in self.ik[rows]]

def find_tier_dir():
    if os.environ.get("CASMI_TIER_DIR"): return os.environ["CASMI_TIER_DIR"]
    hits = glob.glob("/kaggle/input/**/pc_lsid.npy", recursive=True) + glob.glob("data/external/pubchem_tier/pc_lsid.npy")    # dataset root: pc_mass/pc_off/pc_smiles may sit in a subfolder
    return os.path.dirname(hits[0]) if hits else None

_fpg = _taut_ = None
def _fp_of(smi):
    global _fpg
    if _fpg is None: _fpg = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
    m = Chem.MolFromSmiles(smi)
    return None if m is None else _fpg.GetFingerprintAsNumPy(m).astype(np.uint8)
def _metric_key(smi):
    global _taut_
    if _taut_ is None: _taut_ = rdMolStandardize.TautomerEnumerator()
    try:
        m = Chem.MolFromSmiles(smi)
        return None if m is None else Chem.MolToInchiKey(_taut_.Canonicalize(m))[:14]
    except Exception: return None

def _pmap(fn, items, pool):
    items = list(items)
    if pool is None or len(items) < 400: return [fn(x) for x in items]
    return pool.map(fn, items, chunksize=max(50, len(items) // (4 * pool._processes)))

def rank_with_tier(Q, lo, hi, lib, tier, w, k=25, topn=60, fp_cache=None, key_cache=None, pool=None, lib_keyset=None, lib_bonus=0.0, progress=0):
    """Per query: library candidates (mass window) + PubChem tier candidates (same window), scored z(cosine) + w * pop, top `topn`, deduplicated on the
    metric key, first k kept. Returns, per query, a list of (metric_key, smiles, source) with source 'lib' or 'tier'.
    lib_bonus: SOURCE PRIOR added to every library (train + COCONUT) candidate's score: a structure we have seen is a priori far more likely to be the answer
    than one of the thousands of same-mass PubChem isomers (z-score units, like w * pop)."""
    fp_cache = {} if fp_cache is None else fp_cache; key_cache = {} if key_cache is None else key_cache
    lib_keyset = set(lib.keys) if lib_keyset is None else lib_keyset
    out = []
    t_start = time.time()
    for i in range(len(Q)):
        if progress and i and i % progress == 0: tick(f"  PubChem channel: {i:,}/{len(Q):,} molecules ({(time.time() - t_start) / i:.2f} s each, {len(fp_cache):,} tier structures fingerprinted)")
        a, b = np.searchsorted(lib.mass_sorted, [lo[i], hi[i]]); idx = lib.order[a:b]
        ta, tb = tier.window(lo[i], hi[i]); rows = np.arange(ta, tb)
        if len(rows):
            rows = rows[[ik not in lib_keyset for ik in tier.ik14(rows)]]            # library copy wins
            need = [r for r in rows if r not in fp_cache]
            for r, f in zip(need, _pmap(_fp_of, tier.smiles(need), pool)): fp_cache[r] = f
            rows = np.array([r for r in rows if fp_cache[r] is not None], dtype=np.int64)
        if len(idx) == 0 and len(rows) == 0:                                          # nothing at all: widen the library window as rank_queries does
            c = 0.5 * (lo[i] + hi[i])
            for ppm in (10, 25, 50, 100, 250):
                a, b = np.searchsorted(lib.mass_sorted, [c * (1 - ppm * 1e-6), c * (1 + ppm * 1e-6)])
                if b > a: idx = lib.order[a:b]; break
            if len(idx) == 0: idx = lib.order[np.argsort(np.abs(lib.mass_sorted - c))[:k]]
        s_lib = lib.fp_unit[idx] @ Q[i]
        s_tier = unit(np.stack([fp_cache[r] for r in rows]).astype(np.float32)) @ Q[i] if len(rows) else np.zeros(0, np.float32)
        s = np.concatenate([s_lib, s_tier])
        if (w or lib_bonus) and len(s) > 1:
            pop = np.concatenate([lib.pop[idx] if lib.pop is not None else np.zeros(len(idx), np.float32), tier.pop(rows) if len(rows) else np.zeros(0, np.float32)])
            s = (s - s.mean()) / (s.std() + 1e-6) + w * pop
            s[:len(idx)] += lib_bonus
        top = np.argsort(-s)[:topn]
        cand = [("lib", int(idx[j])) if j < len(idx) else ("tier", int(rows[j - len(idx)])) for j in top]
        need = [r for src, r in cand if src == "tier" and r not in key_cache]
        smi_need = tier.smiles(need)
        for r, sm, key in zip(need, smi_need, _pmap(_metric_key, smi_need, pool)): key_cache[r] = (sm, key)
        seen, res = set(), []
        for src, r in cand:
            key, sm = (lib.keys[r], lib.smiles[r]) if src == "lib" else (key_cache[r][1], key_cache[r][0])
            if key is None or key in seen: continue
            seen.add(key); res.append((key, sm, src))
            if len(res) == k: break
        out.append(res)
    return out

def make_submission_tier(P_test, name, tier, w, lib_bonus=0.0, workers=4):
    """Like make_submission, with the PubChem channel."""
    keys, Pm, L, H, nbad = pool_molecules(P_test, test.molecule_id.values, test_lo, test_hi)
    pool = mp.get_context("fork").Pool(workers) if workers > 1 else None
    try: ranked = rank_with_tier(Pm, L, H, LIB, tier, w, pool=pool, lib_bonus=lib_bonus, progress=50)
    finally:
        if pool is not None: pool.close()
    sub = pd.DataFrame({"molecule_id": keys, "smiles": [";".join(sm for _, sm, _ in r) for r in ranked]})
    sub = sub.set_index("molecule_id").loc[sample.molecule_id].reset_index()
    n = validate_submission(sub, sample.molecule_id)
    path = f"{WORK}/submission_{name}{'_debug' if DEBUG else ''}.csv"; sub.to_csv(path, index=False)
    share = np.mean([src == "tier" for r in ranked for _, _, src in r]) if ranked else float("nan")
    tick(f"{path}: {len(sub)} molecules | guesses per molecule: median {n.median():.0f}, min {n.min()} | share of PubChem-tier guesses {share:.1%}")
    return sub, ranked, Pm


In [ ]:
# ======================================== 4. PREDICT + SUBMIT ========================================
keep = np.load(f"{FROZEN_DIR}/keep.npy")
assert X_TEST.shape[1] == META["n_cols_full"], f"test features have {X_TEST.shape[1]} columns, the frozen model expects {META['n_cols_full']}"
net = make_net(len(keep)).to(DEVICE); net.load_state_dict(torch.load(f"{FROZEN_DIR}/deep_best.pt", map_location=DEVICE))
P_test_deep = predict_mlp(net, X_TEST[:, keep].tocsr())
tick(f"predicted {len(P_test_deep):,} test spectra")
if USE_TIER:
    assert POP_WEIGHT > 0, "the PubChem channel needs the popularity prior (POP_WEIGHT > 0): without it thousands of same-mass isomers swamp the library"
    TIER = Tier(find_tier_dir()); tick(f"PubChem tier: {len(TIER.mass):,} structures")
    SUB_DEEP, RANKED_DEEP, _ = make_submission_tier(P_test_deep, "deep", TIER, POP_WEIGHT, lib_bonus=TIER_LIB_BONUS, workers=TIER_WORKERS)
    publish(SUB_DEEP, f"frozen deep MLP + PubChem channel, library {len(LIB.keys):,}, popularity weight {POP_WEIGHT}, library bonus {TIER_LIB_BONUS}")
else:
    SUB_DEEP, RANKED_DEEP, _ = make_submission(P_test_deep, "deep")
    publish(SUB_DEEP, f"frozen deep MLP, library {len(LIB.keys):,}, popularity weight {POP_WEIGHT}")
print("\nexample rows:"); print(SUB_DEEP.assign(smiles=SUB_DEEP.smiles.str.split(";").str[:3].str.join(";")).head(5).to_string(index=False))
tick("done")
